Task : Develop an anomaly detection system to identify unusual patterns or outliers in data.
\
\
Details:
\
Data Preparation: Clean and preprocess data for anomaly detection.
\
Model Selection: Choose and implement anomaly detection algorithms (e.g., Isolation Forest, One-Class SVM,
Autoencoders).
\
Evaluation: Evaluate the model using metrics like precision, recall, and F1 score for anomaly detection.
\
Deployment: Integrate the anomaly detection system into a monitoring framework or application.
\
\
\
Where to Do It:
Scikit-Learn: Use for implementing anomaly detection algorithms.
\
TensorFlow: Develop anomaly detection models using deep learning techniques.
\
Google Colab: Experiment with different anomaly detection methods.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(path + '/creditcard.csv')
df.head()

In [ ]:
df.columns

In [ ]:
df.Class.value_counts()

In [ ]:
492/(284315+492)

In [ ]:
df.duplicated().any()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().any()

In [ ]:
df.isna().sum()

In [ ]:
df.isna().any().sum()

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report



class NoDuplicates(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    self.feature_names_in_ = X.columns.tolist()
    return self

  def transform(self, X):
    return X.drop_duplicates()

  def get_feature_names_out(self, input_features=None):
    if input_features is None:
      input_features = self.feature_names_in_
    return input_features





class DropTime(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    self.feature_names_in_ = X.columns.to_list()
    return self

  def transform(self, X):
    if 'Time' in X.columns:
      return X.drop('Time', axis=1)
    return X

  def get_feature_names_out(self, input_features=None):
    if input_features is None:
      input_features = self.feature_names_in_
    return list(set(input_features).difference({'Time'}))




pipe = Pipeline([
    ('drop', DropTime()),
    ('nodup', NoDuplicates()),
])


df_cleaned = pd.DataFrame(pipe.fit_transform(df), columns=pipe.get_feature_names_out())

X = df_cleaned.drop('Class', axis=1)
y = df_cleaned['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, stratify=y_test, random_state=42)



logreg_pipe = Pipeline([
    ('scale', RobustScaler()),
    ('logreg', LogisticRegression(class_weight={0: 1, 1: 20}, max_iter=1_000))
])


isof_pipe = Pipeline([
    ('scale', RobustScaler()),
    ('isof', IsolationForest(contamination=0.002, n_jobs=-1))
])



outlier_methods = {
    'Logistic Regression': logreg_pipe,
    'Isolation Forest': isof_pipe
}


params_grid = {
    'Logistic Regression': {'logreg__class_weight': [{0:1, 1:20}, {0:1, 1:18}, {0:1, 1:16}, {0:1, 1:14}, {0:1, 1:12}, {0:1, 1:10}]},
    'Isolation Forest': {'isof__contamination': [0.0001, 0.0007, 0.001, 0.002, 0.004, 0.01, 0.02, 0.04, 0.1]}
}

logreg_f1_val = []
isof_f1_val = []

print(f'X_train shape : {X_train.shape}, X_val shape : {X_val.shape}')

for name, model in outlier_methods.items():
  print(f'{name} model :')
  if name == 'Logistic Regression':
    for class_weights in params_grid[name]['logreg__class_weight']:
      print('{} model with class weights : {}'.format(name, class_weights))
      model.steps[1][1].class_weight = class_weights
      model.fit(X_train, y_train)
      predictions = model.predict(X_val)
      logreg_f1_val.append(f1_score(y_val, predictions))
      print('validation f1 score : {}'.format(f1_score(y_val, predictions)))
      print('validation precision score : {}'.format(precision_score(y_val, predictions)))
      print('validation recall score : {}'.format(recall_score(y_val, predictions)))
      print('\n')
      print(classification_report(y_val, predictions))
      print('\n'*2)

  elif name == 'Isolation Forest':
    for contaminations in params_grid[name]['isof__contamination']:
      print('{} model with contamination : {}'.format(name, contaminations))
      model.steps[1][1].contamination = contaminations
      model.fit(X_train)
      predictions = model.predict(X_val)
      print(('{} model predicts : {} outliers for validation data. In reality, validation data has {} outliers'.\
             format(name, (predictions == -1).sum(), (y_val==1).sum())))
      print('\n'*2)

      predictions_processed_for_classification_report = (predictions==-1).astype(int)
      isof_f1_val.append(f1_score(y_val, predictions_processed_for_classification_report))

      print('validation f1 score : {}'.format(f1_score(y_val, predictions_processed_for_classification_report)))
      print('validation precision score : {}'.format(precision_score(y_val, predictions_processed_for_classification_report)))
      print('validation recall score : {}'.format(recall_score(y_val, predictions_processed_for_classification_report)))
      print('\n')
      print(classification_report(y_val, predictions_processed_for_classification_report))
      print('\n'*2)




print('\n'*5)

print('Best class_weights value for Logistic Regression: {}, best f1 averaged: {}'\
      .format(params_grid.get('Logistic Regression').get('logreg__class_weight')[logreg_f1_val.index(max(logreg_f1_val))], max(logreg_f1_val)))

print('Best contamination value for Isolation Forest Model: {}, best f1 averaged: {}'\
      .format(params_grid.get('Isolation Forest').get('isof__contamination')[isof_f1_val.index(max(isof_f1_val))], max(isof_f1_val)))


In [ ]:
logreg_f1_val.index(max(logreg_f1_val))

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(0.99)

X_reduced = pca.fit_transform(X)

X_reduced.shape

In [ ]:
X_reduced

In [ ]:
pca = PCA(n_components=3)

X3D = pca.fit_transform(X)


fig = plt.figure()
ax = plt.axes(projection='3d')
ax.scatter(X3D[:, 0], X3D[:, 1], X3D[:, 2])
plt.show()

In [ ]:
df_no_fraud = df_cleaned[df_cleaned['Class']==0].sample((df_cleaned['Class']==1).sum())
df_fraud = df_cleaned[df_cleaned['Class']==1]

df_cleaned_balanced = pd.concat([df_no_fraud, df_fraud], axis=0)

df_cleaned_balanced

In [ ]:
logreg = LogisticRegression(max_iter=1_000)
logreg.fit(df_cleaned_balanced.drop('Class', axis=1), df_cleaned_balanced['Class'])
print(classification_report(y, logreg.predict(X)))